# Actividad: Del dato crudo al dato de negocio

## Construcción de una arquitectura Medallion en Databricks

En esta actividad se construira una arquitectura de datos **Medallion** utilizando Databricks Free Edition, Unity Catalog, Python, SQL y tablas Delta.

Se trabajará con un conjunto de datos de películas descargado de Internet. El archivo contiene información sobre títulos, fechas de estreno, géneros, presupuestos, recaudaciones y calificaciones.

El objetivo es transformar los datos originales en información fiable y útil para el negocio, aplicando reglas de calidad y mecanismos de gobierno.

## Objetivos

- Crear las capas Bronze, Silver y Gold en Unity Catalog.
- Descargar y conservar los datos originales.
- Identificar errores, valores nulos y posibles duplicados.
- Aplicar reglas de calidad y documentar los rechazos.
- Crear tablas analíticas para responder preguntas de negocio.
- Gestionar comentarios, etiquetas, permisos y trazabilidad.

## Arquitectura Medallion

### 🥉 Bronze: datos originales

Conserva los datos recibidos del origen sin modificaciones importantes, para poder consultarlos y recuperarlos cuando sea necesario.

### 🥈 Silver: datos limpios y validados

Limpia, organiza y transforma los datos. Corrige los tipos de datos, identifica valores nulos y duplicados y aplica reglas de calidad.

### 🥇 Gold: datos preparados para el negocio

Ofrece información agregada y preparada para el análisis, facilitando la creación de informes y la toma de decisiones.

## Resultado esperado

Al finalizar, podremos seguir el recorrido de los datos desde el archivo original hasta las tablas utilizadas por los analistas, garantizando una mayor calidad, organización y trazabilidad de la información.


# 1. Preparar el entorno

Antes de cargar los datos, se debe de preparar la estructura de almacenamiento.

Se utilizará `workspace`, que contiene los tres esquemas de nuestra arquitectura:

- `cine_bronze`
- `cine_silver`
- `cine_gold`

Separar las capas permite organizar el trabajo y establecer permisos diferentes según las necesidades de cada usuario.

También se crea un volumen en Bronze.

## Resultado esperado

Tres esquemas y un volumen de aterrizaje.

In [0]:
ESQUEMAS = {
    "cine_bronze": "Capa Bronze: datos originales",
    "cine_silver": "Capa Silver: datos limpios y reglas de calidad",
    "cine_gold": "Capa Gold: tablas preparadas para negocio",
}

for nombre, descripcion in ESQUEMAS.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS workspace.{nombre}
        COMMENT '{descripcion}'
    """)
    print("Esquema listo:", nombre)

Esquema listo: cine_bronze
Esquema listo: cine_silver
Esquema listo: cine_gold


In [0]:
%sql
SELECT schema_name, comment
FROM workspace.information_schema.schemata
WHERE schema_name LIKE 'cine_%'
ORDER BY schema_name;

schema_name,comment
cine_bronze,"Capa Bronze: datos tal como llegaron del origen, sin transformar"
cine_gold,Capa Gold: tablas agregadas listas para consumo de negocio
cine_silver,"Capa Silver: datos limpios, tipados y con reglas de calidad"


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.cine_bronze.aterrizaje
COMMENT 'Zona de aterrizaje para los archivos originales';


# 2. Descargar los datos originales

Se utiliza el conjunto de datos `movies.json`, publicado en el repositorio público de `vega-datasets`.

El archivo contiene información de películas y algunos campos incompletos o inconsistentes, lo que permite practicar técnicas de perfilado y validación.

Luego se guarda el archivo en el volumen de Bronze sin modificar su contenido.

Y por último se conserva el original porque permite revisar el dato recibido y reconstruir el proceso de transformación si fuera necesario.

## Resultado esperado

El archivo `movies.json` almacenado en el volumen de aterrizaje.

In [0]:
import urllib.request

URL = "https://raw.githubusercontent.com/vega/vega-datasets/main/data/movies.json"
DESTINO = "/Volumes/workspace/cine_bronze/aterrizaje/movies.json"

with urllib.request.urlopen(URL, timeout=60) as respuesta:
    contenido = respuesta.read()

with open(DESTINO, "wb") as f:
    f.write(contenido)

print(f"Descargados {len(contenido):,} bytes")
print("Guardado en:", DESTINO)

Descargados 1,399,981 bytes
Guardado en: /Volumes/workspace/cine_bronze/aterrizaje/movies.json


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/cine_bronze/aterrizaje"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/cine_bronze/aterrizaje/movies.json,movies.json,1399981,1792765295000



# 3. Capa Bronze: conservar lo que llegó

La capa Bronze es la representación inicial de los datos recibidos.

En esta etapa no se corrigen fechas, ni se eliminan registros ni se modifican los valores de negocio. El objetivo es preservar el contenido original y registrar información de trazabilidad.

El archivo contiene nombres de columnas con espacios, como `US Gross` y `Major Genre`. Para facilitar su almacenamiento en Delta, se normalizan los nombres a minúsculas y se sustituyen los caracteres especiales por guiones bajos.

## Metadatos añadidos

Se añadieron dos metadatos:

- `_archivo_origen`: identifica el archivo del que procede la información.
- `_ingestado_en`: registra el momento de la ingesta.

## Resultado esperado

Una tabla Delta llamada `workspace.cine_bronze.peliculas_crudo`.

In [0]:
import re
from pyspark.sql import functions as F

df_crudo = (
    spark.read
    .option("multiLine", True)
    .json(DESTINO)
)

print("Filas originales:", df_crudo.count())
df_crudo.printSchema()

def sanear(nombre):
    return re.sub(r"[^0-9a-zA-Z]+", "_", nombre).strip("_").lower()

df_con_metadatos = (
    df_crudo
    .withColumn("_archivo_origen", F.lit(DESTINO))
    .withColumn("_ingestado_en", F.current_timestamp())
)

nombres = [
    sanear(c) for c in df_crudo.columns
] + ["_archivo_origen", "_ingestado_en"]

df_bronze = df_con_metadatos.toDF(*nombres)

Filas originales: 3201
root
 |-- Creative Type: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Distributor: string (nullable = true)
 |-- IMDB Rating: double (nullable = true)
 |-- IMDB Votes: long (nullable = true)
 |-- MPAA Rating: string (nullable = true)
 |-- Major Genre: string (nullable = true)
 |-- Production Budget: long (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- Rotten Tomatoes Rating: long (nullable = true)
 |-- Running Time min: long (nullable = true)
 |-- Source: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- US DVD Sales: long (nullable = true)
 |-- US Gross: long (nullable = true)
 |-- Worldwide Gross: long (nullable = true)



In [0]:
(
    df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("workspace.cine_bronze.peliculas_crudo")
)

In [0]:
%sql
SELECT *
FROM workspace.cine_bronze.peliculas_crudo
LIMIT 10;

creative_type,director,distributor,imdb_rating,imdb_votes,mpaa_rating,major_genre,production_budget,release_date,rotten_tomatoes_rating,running_time_min,source,title,us_dvd_sales,us_gross,worldwide_gross,_archivo_origen,_ingestado_en
null,null,Gramercy,6.1,1071,R,null,8000000,Jun 12 1998,null,null,null,The Land Girls,null,146083,146083,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,null,Strand,6.9,207,R,Drama,300000,Aug 07 1998,null,null,null,"First Love, Last Rites",null,10876,10876,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,null,Lionsgate,6.8,865,null,Comedy,250000,Aug 28 1998,null,null,null,I Married a Strange Person,null,203134,203134,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,null,Fine Line,null,null,null,Comedy,300000,Sep 11 1998,13,null,null,Let's Talk About Sex,null,373615,373615,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
Contemporary Fiction,null,Trimark,3.4,165,R,Drama,1000000,Oct 09 1998,62,null,Original Screenplay,Slam,null,1009819,1087521,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,null,MGM,null,null,null,null,1600000,Jan 15 1999,null,null,null,Mississippi Mermaid,null,24551,2624551,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,Christopher Nolan,Zeitgeist,7.7,15133,R,null,6000,Apr 04 1999,null,null,null,Following,null,44705,44705,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
Contemporary Fiction,null,Artisan,3.8,353,R,Comedy,1600000,Apr 09 1999,null,null,Original Screenplay,Foolish,null,6026908,6026908,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,Roman Polanski,null,5.8,3275,R,null,40000000,Jul 01 1986,25,null,null,Pirates,null,1641825,6341825,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z
null,null,null,7.0,2906,null,null,6000000,Dec 31 2046,86,null,null,Duel in the Sun,null,20400000,20400000,/Volumes/workspace/cine_bronze/aterrizaje/movies.json,2026-10-23T14:24:23.353Z



# 4. Perfilado: conocer antes de limpiar

Antes de establecer reglas de calidad debemos observar el estado real de los datos.

Se debe se calcular el porcentaje de valores nulos de cada columna. Esto permite distinguir entre campos que pueden utilizarse directamente y campos que necesitan un tratamiento especial.

## Problemas que investigaremos

- **Fechas de estreno en años futuros:** películas cuyas fechas de estreno sean posteriores a la fecha actual.
- **Títulos repetidos:** títulos que aparecen más de una vez y comprobaremos si realmente son duplicados.

No se debe de asumir que todos los títulos repetidos son duplicados. Una nueva versión de una película puede compartir título con una producción anterior.

## Criterio de trabajo

Las reglas de limpieza se definirán a partir de los problemas detectados, no de suposiciones.

In [0]:
tabla = spark.table("workspace.cine_bronze.peliculas_crudo")
total = tabla.count()

nulos = tabla.select([
    F.round(
        100 * F.sum(F.col(c).isNull().cast("int")) / total,
        1
    ).alias(c)
    for c in tabla.columns
])

display(nulos)

creative_type,director,distributor,imdb_rating,imdb_votes,mpaa_rating,major_genre,production_budget,release_date,rotten_tomatoes_rating,running_time_min,source,title,us_dvd_sales,us_gross,worldwide_gross,_archivo_origen,_ingestado_en
13.9,41.6,7.2,6.7,6.7,18.9,8.6,0.0,0.0,27.5,62.2,11.4,0.0,82.4,0.2,0.2,0.0,0.0


In [0]:
%sql
SELECT
    right(release_date, 4) AS anio,
    count(*) AS peliculas
FROM workspace.cine_bronze.peliculas_crudo
GROUP BY ALL
ORDER BY anio DESC
LIMIT 15;

anio,peliculas
2046,2
2044,1
2043,1
2042,1
2041,1
2040,1
2039,2
2037,1
2036,2
2033,3


In [0]:
%sql
SELECT title, count(*) AS veces
FROM workspace.cine_bronze.peliculas_crudo
GROUP BY title
HAVING count(*) > 1
ORDER BY veces DESC, title
LIMIT 10;

title,veces
"20,000 Leagues Under the Sea",2
A Nightmare on Elm Street,2
Alice in Wonderland,2
Around the World in 80 Days,2
Ben-Hur,2
Casino Royale,2
Crash,2
Dawn of the Dead,2
Day of the Dead,2
Death at a Funeral,2



# 5. Capa Silver: tipar, validar y documentar

En Silver se transforman los datos originales en una tabla preparada para el análisis.

Se aplican tres tipos de tratamiento:

## 1. Tipado y nombres de negocio

Se debe de convertir las fechas a tipo `DATE`, los importes a tipos numéricos y utilizaremos nombres de columnas comprensibles en español.

## 2. Reglas de calidad

Se rechazan los registros que no cumplan las condiciones mínimas establecidas:

- El título debe existir.
- La fecha debe poder interpretarse y no ser futura.
- El presupuesto debe ser positivo.

## 3. Cuarentena

No se elimina silenciosamente los registros rechazados. Se guardan en una tabla independiente junto con los motivos del rechazo.

Para evitar eliminar remakes legítimos, la clave de negocio será la combinación de título y fecha de estreno.

## Resultado esperado

Una tabla Silver de películas válidas y otra tabla de cuarentena con los registros rechazados.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_tipadas AS
SELECT
    title AS titulo,
    try_to_date(release_date, 'MMM dd yyyy') AS fecha_estreno,
    major_genre AS genero,
    mpaa_rating AS clasificacion,
    distributor AS estudio,
    director,
    creative_type AS tipo_creativo,
    source AS origen_historia,
    CAST(running_time_min AS INT) AS duracion_min,
    CAST(production_budget AS BIGINT) AS presupuesto,
    nullif(CAST(us_gross AS BIGINT), 0) AS recaudacion_eeuu,
    nullif(CAST(worldwide_gross AS BIGINT), 0) AS recaudacion_mundial,
    CAST(imdb_rating AS DOUBLE) AS calificacion_imdb,
    CAST(imdb_votes AS BIGINT) AS votos_imdb,
    CAST(rotten_tomatoes_rating AS INT) AS calificacion_rt,
    _ingestado_en
FROM workspace.cine_bronze.peliculas_crudo;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_evaluadas AS
SELECT *,
    array_compact(array(
        CASE WHEN titulo IS NULL
            THEN 'titulo nulo' END,
        CASE WHEN fecha_estreno IS NULL
            THEN 'fecha ilegible' END,
        CASE WHEN fecha_estreno > current_date()
            THEN 'fecha futura' END,
        CASE WHEN presupuesto IS NULL OR presupuesto <= 0
            THEN 'presupuesto no positivo' END
    )) AS motivos_rechazo
FROM peliculas_tipadas;

In [0]:
%sql
SELECT motivo, count(*) AS filas
FROM (
    SELECT explode(motivos_rechazo) AS motivo
    FROM peliculas_evaluadas
    WHERE size(motivos_rechazo) > 0
)
GROUP BY motivo
ORDER BY filas DESC;

motivo,filas
fecha futura,16
presupuesto no positivo,1
titulo nulo,1


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas_cuarentena
COMMENT 'Filas que no superaron las reglas de calidad de Silver'
AS
SELECT *, current_timestamp() AS _cuarentena_en
FROM peliculas_evaluadas
WHERE size(motivos_rechazo) > 0;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas
COMMENT 'Películas limpias y tipadas'
AS
WITH validas AS (
    SELECT *
    FROM peliculas_evaluadas
    WHERE size(motivos_rechazo) = 0
),
clasificadas AS (
    SELECT *,
        row_number() OVER (
            PARTITION BY titulo, fecha_estreno
            ORDER BY recaudacion_mundial DESC NULLS LAST
        ) AS rn
    FROM validas
)
SELECT
    titulo, fecha_estreno, genero, clasificacion, estudio,
    director, tipo_creativo, origen_historia, duracion_min,
    presupuesto, recaudacion_eeuu, recaudacion_mundial,
    calificacion_imdb, votos_imdb, calificacion_rt,
    _ingestado_en
FROM clasificadas
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:

%sql
SELECT
    (SELECT count(*) FROM workspace.cine_bronze.peliculas_crudo) AS bronze,
    (SELECT count(*) FROM workspace.cine_silver.peliculas) AS silver,
    (SELECT count(*) FROM workspace.cine_silver.peliculas_cuarentena) AS cuarentena,
    (SELECT count(*) FROM workspace.cine_silver.peliculas)
      + (SELECT count(*) FROM workspace.cine_silver.peliculas_cuarentena)
      AS suma_comprobacion;

bronze,silver,cuarentena,suma_comprobacion
3201,3183,18,3201



# 6. Contrato de datos en Silver

Las reglas de calidad no deberían depender únicamente del código que carga los datos. También se debe de proteger la tabla frente a inserciones incorrectas realizadas por otros procesos.

Para ello, se establecen restricciones que impidan almacenar:

- Presupuestos no positivos.
- Duraciones fuera de rango.
- Calificaciones de IMDb inválidas.

Además, se registra la clave de negocio formada por el título y la fecha de estreno.

## Consideraciones sobre Unity Catalog

En Unity Catalog, la clave primaria sirve como metadato informativo; las restricciones `CHECK` son las que validan las condiciones que especifican.

## Resultado esperado

Una tabla protegida por restricciones y con su clave de negocio documentada.

In [0]:
CONTRATO = [
    "ALTER COLUMN titulo SET NOT NULL",
    "ALTER COLUMN fecha_estreno SET NOT NULL",
    "ADD CONSTRAINT presupuesto_positivo CHECK (presupuesto > 0)",
    "ADD CONSTRAINT duracion_razonable CHECK (duracion_min IS NULL OR duracion_min BETWEEN 1 AND 500)",
    "ADD CONSTRAINT imdb_en_rango CHECK (calificacion_imdb IS NULL OR calificacion_imdb BETWEEN 0 AND 10)",
]

for clausula in CONTRATO:
    spark.sql(
        f"ALTER TABLE workspace.cine_silver.peliculas {clausula}"
    )

In [0]:
%sql
ALTER TABLE workspace.cine_silver.peliculas
ADD CONSTRAINT pk_peliculas
PRIMARY KEY (titulo, fecha_estreno);


# 7. Capa Gold: responder preguntas de negocio

Gold contiene información preparada para su consumo por analistas y herramientas de visualización.

A diferencia de Silver, que conserva el detalle de cada película, Gold organiza los datos para responder preguntas concretas.

## Tablas analíticas

Para ello se construyen dos tablas:

### 1. `resumen_genero_anio`

Se comparan los siguientes indicadores por año y género:

- Número de películas.
- Recaudación.
- Presupuesto.
- Retorno medio.
- Calificación de IMDb.

### 2. `ranking_estudios`

Se comparan los estudios con al menos veinte películas, ordenados por recaudación mundial acumulada.

## Vista pública

También se crea una vista pública que permita consultar información general sin exponer las columnas financieras.

## Resultado esperado

Tablas analíticas listas para ser consultadas por el negocio.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_gold.resumen_genero_anio
COMMENT 'Volumen, taquilla, retorno y nota media por año y género'
AS
SELECT
    year(fecha_estreno) AS anio,
    coalesce(genero, 'Sin clasificar') AS genero,
    count(*) AS peliculas,
    sum(recaudacion_mundial) AS recaudacion_mundial,
    sum(presupuesto) AS presupuesto,
    round(avg(try_divide(recaudacion_mundial, presupuesto)), 2)
        AS retorno_medio,
    round(avg(calificacion_imdb), 2) AS nota_imdb_media
FROM workspace.cine_silver.peliculas
GROUP BY year(fecha_estreno), coalesce(genero, 'Sin clasificar');

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_gold.ranking_estudios
COMMENT 'Estudios con al menos 20 películas, ordenados por taquilla'
AS
SELECT
    estudio,
    count(*) AS peliculas,
    sum(recaudacion_mundial) AS recaudacion_mundial,
    round(avg(calificacion_imdb), 2) AS nota_imdb_media,
    rank() OVER (
        ORDER BY sum(recaudacion_mundial) DESC
    ) AS puesto
FROM workspace.cine_silver.peliculas
WHERE estudio IS NOT NULL
GROUP BY estudio
HAVING count(*) >= 20;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE VIEW workspace.cine_gold.catalogo_publico
COMMENT 'Catálogo de películas sin información financiera'
AS
SELECT
    titulo, fecha_estreno, genero, clasificacion, estudio,
    director, duracion_min, calificacion_imdb, calificacion_rt
FROM workspace.cine_silver.peliculas;


# 8. Gobierno de datos con Unity Catalog

La calidad técnica no es suficiente para garantizar un uso correcto de la información. También se necesita que los usuarios sepan qué significan las columnas, qué datos son sensibles, quién tiene acceso y qué operaciones se han realizado.

## Mecanismos de gobierno

En esta sección se aplican cuatro mecanismos de gobierno:

### 1. Documentación
Se añaden comentarios en las tablas y columnas para explicar su contenido y significado.

### 2. Clasificación
Se aplican etiquetas para identificar la capa a la que pertenece cada tabla y la sensibilidad de los datos.

### 3. Permisos
Se limita el acceso según las necesidades de los usuarios y sus responsabilidades.

### 4. Trazabilidad
Se consulta el inventario de objetos y el historial de operaciones para conocer los cambios realizados.

## Vista pública de Gold

La vista pública de Gold permitirá ilustrar cómo se puede ofrecer un conjunto de columnas de uso general sin exponer directamente los importes financieros.

## Resultado esperado

Tablas documentadas, etiquetas aplicadas, permisos comprobados e historial disponible.

In [0]:
COMENTARIOS = {
    "titulo": "Título original de la película",
    "fecha_estreno": "Fecha de estreno en salas",
    "presupuesto": "Presupuesto de producción en dólares. Dato confidencial",
    "recaudacion_mundial": "Taquilla mundial en dólares",
    "calificacion_imdb": "Nota media de IMDb, de 0 a 10",
}

for columna, texto in COMENTARIOS.items():
    texto_sql = texto.replace("'", "''")
    spark.sql(f"""
        ALTER TABLE workspace.cine_silver.peliculas
        ALTER COLUMN {columna} COMMENT '{texto_sql}'
    """)

In [0]:
%sql
ALTER TABLE workspace.cine_silver.peliculas
SET TAGS (
    'capa' = 'silver',
    'dominio' = 'cine',
    'certificada' = 'si'
);


ALTER TABLE workspace.cine_silver.peliculas
ALTER COLUMN presupuesto
SET TAGS ('sensibilidad' = 'confidencial');


ALTER TABLE workspace.cine_gold.resumen_genero_anio
SET TAGS ('capa' = 'gold', 'dominio' = 'cine');

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA workspace.cine_gold
TO `account users`;


GRANT SELECT
ON SCHEMA workspace.cine_gold
TO `account users`;

In [0]:
%sql
SELECT
    table_schema AS capa,
    table_name AS tabla,
    table_type AS tipo,
    comment
FROM workspace.information_schema.tables
WHERE table_schema IN ('cine_bronze', 'cine_silver', 'cine_gold')
ORDER BY capa, tabla;

capa,tabla,tipo,comment
cine_bronze,peliculas_crudo,MANAGED,null
cine_gold,catalogo_publico,VIEW,Catálogo de películas sin información financiera
cine_gold,ranking_estudios,MANAGED,"Estudios con al menos 20 películas, ordenados por taquilla"
cine_gold,resumen_genero_anio,MANAGED,"Volumen, taquilla, retorno y nota media por año y género"
cine_silver,peliculas,MANAGED,Películas limpias y tipadas
cine_silver,peliculas_cuarentena,MANAGED,Filas que no superaron las reglas de calidad de Silver


In [0]:
hist = (
    spark.sql("DESCRIBE HISTORY workspace.cine_silver.peliculas")
    .select("version", "timestamp", "operation")
    .orderBy("version")
)

display(hist)

version,timestamp,operation
0,2026-10-23T09:58:42.000Z,CREATE OR REPLACE TABLE AS SELECT
1,2026-10-23T14:26:39.000Z,CREATE OR REPLACE TABLE AS SELECT
2,2026-10-23T14:28:19.000Z,CHANGE COLUMN
3,2026-10-23T14:28:21.000Z,CHANGE COLUMN
4,2026-10-23T14:28:24.000Z,ADD CONSTRAINT
5,2026-10-23T14:28:26.000Z,ADD CONSTRAINT
6,2026-10-23T14:28:27.000Z,ADD CONSTRAINT
7,2026-10-23T14:31:17.000Z,CHANGE COLUMN
8,2026-10-23T14:31:18.000Z,CHANGE COLUMN
9,2026-10-23T14:31:19.000Z,CHANGE COLUMN



# Actividad Práctica 01: retos adicionales

Esta sección completa los tres retos solicitados en la actividad práctica. Ejecuta las celdas en orden en el mismo entorno de Databricks donde se construyeron las capas Bronze, Silver y Gold.

> Importante: los resultados numéricos dependen de la ejecución real en tu workspace. Guarda las salidas de las consultas para documentarlas en el Word.


## Reto 1. Crear la tabla Gold peliculas_rentables

Se calcula el retorno como la taquilla mundial dividida entre el presupuesto. try_divide evita errores de división y el filtro exige más de 10 000 votos en IMDb. Se ordena de mayor a menor retorno y se conservan las 50 primeras películas.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_gold.peliculas_rentables
COMMENT 'Las 50 películas con mayor retorno taquilla mundial/presupuesto entre títulos con más de 10000 votos en IMDb'
AS
SELECT
    titulo,
    fecha_estreno,
    genero,
    director,
    presupuesto,
    recaudacion_mundial,
    votos_imdb,
    calificacion_imdb,
    try_divide(recaudacion_mundial, presupuesto) AS retorno
FROM workspace.cine_silver.peliculas
WHERE votos_imdb > 10000
  AND presupuesto > 0
  AND recaudacion_mundial IS NOT NULL
ORDER BY retorno DESC
LIMIT 50;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.cine_gold.peliculas_rentables
ORDER BY retorno DESC;

titulo,fecha_estreno,genero,director,presupuesto,recaudacion_mundial,votos_imdb,calificacion_imdb,retorno
Paranormal Activity,2009-09-25,Horror,Oren Peli,15000,193770453,53455,6.7,12918.0302
Mad Max,1980-03-21,Action,George Miller,200000,99750000,36548,6.9,498.75
Super Size Me,2004-05-07,Documentary,Morgan Spurlock,65000,29529368,33805,7.6,454.29796923076924
The Blair Witch Project,1999-07-14,Horror,null,600000,248300000,87629,6.2,413.8333333333333
El Mariachi,1993-02-26,Action,Robert Rodriguez,7000,2040920,19668,7.0,291.56
Night of the Living Dead,1968-10-01,Horror,null,114000,30000000,10083,6.6,263.1578947368421
Halloween,1978-10-17,Horror,John Carpenter,325000,70000000,39866,6.0,215.3846153846154
The Texas Chainsaw Massacre,1974-10-18,Horror,Tobe Hooper,140000,26572439,39172,6.1,189.8031357142857
American Graffiti,1973-08-11,Comedy,George Lucas,777000,140000000,30952,7.6,180.18018018018017
Once,2007-06-16,Drama,null,150000,18997174,34348,8.0,126.64782666666666


In [0]:
%sql
SELECT COUNT(*) AS total_peliculas_rentables
FROM workspace.cine_gold.peliculas_rentables;

total_peliculas_rentables
50



## Reto 2. Añadir una regla de calidad

La nueva regla rechaza las películas cuando género y director están ambos ausentes. Primero se guarda el número actual de filas en cuarentena como línea base. Después se redefine peliculas_evaluadas con la regla nueva y se vuelve a materializar la cuarentena. La diferencia permite calcular cuántas filas adicionales se enviaron a cuarentena.

Ejecuta las siguientes celdas en orden. La celda que crea la vista vuelve a evaluar todas las reglas de calidad desde el punto 5.1.

In [0]:
filas_cuarentena_antes = spark.table(
    "workspace.cine_silver.peliculas_cuarentena"
).count()

print("Filas en cuarentena antes de la nueva regla:", filas_cuarentena_antes)

Filas en cuarentena antes de la nueva regla: 18


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_evaluadas AS
SELECT *,
    array_compact(array(
        CASE WHEN titulo IS NULL
            THEN 'titulo nulo' END,
        CASE WHEN fecha_estreno IS NULL
            THEN 'fecha ilegible' END,
        CASE WHEN fecha_estreno > current_date()
            THEN 'fecha futura' END,
        CASE WHEN presupuesto IS NULL OR presupuesto <= 0
            THEN 'presupuesto no positivo' END,
        CASE WHEN genero IS NULL AND director IS NULL
            THEN 'sin género ni director' END
    )) AS motivos_rechazo
FROM peliculas_tipadas;

In [0]:
%sql
SELECT motivo, COUNT(*) AS filas
FROM (
    SELECT EXPLODE(motivos_rechazo) AS motivo
    FROM peliculas_evaluadas
    WHERE SIZE(motivos_rechazo) > 0
)
GROUP BY motivo
ORDER BY filas DESC;

motivo,filas
sin género ni director,167
fecha futura,16
presupuesto no positivo,1
titulo nulo,1


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas_cuarentena
COMMENT 'Filas que no superaron las reglas de calidad de Silver'
AS
SELECT *, current_timestamp() AS _cuarentena_en
FROM peliculas_evaluadas
WHERE SIZE(motivos_rechazo) > 0;

num_affected_rows,num_inserted_rows


In [0]:
filas_cuarentena_despues = spark.table(
    "workspace.cine_silver.peliculas_cuarentena"
).count()

filas_adicionales_cuarentena = (
    filas_cuarentena_despues - filas_cuarentena_antes
)

print("Filas antes:", filas_cuarentena_antes)
print("Filas después:", filas_cuarentena_despues)
print("Filas adicionales en cuarentena:", filas_adicionales_cuarentena)

Filas antes: 18
Filas después: 180
Filas adicionales en cuarentena: 162



Nota de ejecución: la tabla workspace.cine_silver.peliculas también debe reconstruirse desde la vista peliculas_evaluadas para que refleje la nueva regla. Ejecuta la consulta siguiente y, si tu entorno exige las restricciones del apartado 6, vuelve a aplicar después las restricciones/clave documentadas en ese apartado.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas
COMMENT 'Películas limpias y tipadas tras aplicar las reglas de calidad'
AS
WITH validas AS (
    SELECT *
    FROM peliculas_evaluadas
    WHERE SIZE(motivos_rechazo) = 0
),
clasificadas AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY titulo, fecha_estreno
            ORDER BY recaudacion_mundial DESC NULLS LAST
        ) AS rn
    FROM validas
)
SELECT
    titulo, fecha_estreno, genero, clasificacion, estudio,
    director, tipo_creativo, origen_historia, duracion_min,
    presupuesto, recaudacion_eeuu, recaudacion_mundial,
    calificacion_imdb, votos_imdb, calificacion_rt,
    _ingestado_en
FROM clasificadas
WHERE rn = 1;

num_affected_rows,num_inserted_rows



## Reto 3. Documentar, etiquetar y conceder permisos

Se documenta la tabla Gold con COMMENT, se añaden etiquetas con SET TAGS y se concede permiso de lectura mediante GRANT SELECT. El permiso USE SCHEMA permite a los usuarios acceder al esquema. Si el grupo o principal autorizado en tu workspace tiene otro nombre, sustituye account users por el principal correcto.

In [0]:
%sql
COMMENT ON TABLE workspace.cine_gold.peliculas_rentables IS
'Top 50 de películas con mayor retorno de taquilla mundial sobre presupuesto, considerando únicamente películas con más de 10000 votos en IMDb.';

In [0]:
%sql
ALTER TABLE workspace.cine_gold.peliculas_rentables
SET TAGS (
    'capa' = 'gold',
    'dominio' = 'cine',
    'uso' = 'analitica',
    'criterio' = 'retorno_peliculas'
);

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA workspace.cine_gold
TO `account users`;

GRANT SELECT
ON TABLE workspace.cine_gold.peliculas_rentables
TO `account users`;

In [0]:
%sql
SELECT
    table_schema AS esquema,
    table_name AS tabla,
    comment
FROM workspace.information_schema.tables
WHERE table_schema = 'cine_gold'
  AND table_name = 'peliculas_rentables';

esquema,tabla,comment
cine_gold,peliculas_rentables,"Top 50 de películas con mayor retorno de taquilla mundial sobre presupuesto, considerando únicamente películas con más de 10000 votos en IMDb."



## Conclusiones de la actividad práctica

- La tabla Gold peliculas_rentables presenta las 50 películas con mayor retorno entre las que cumplen el umbral de votos de IMDb.
- La regla de calidad añadida envía a cuarentena las películas que no tienen ni género ni director.
- El incremento de filas en cuarentena se obtiene comparando el recuento anterior y posterior a la nueva regla; debe registrarse a partir de la salida de la consulta.
- La tabla se documenta y clasifica con metadatos de Unity Catalog y se concede permiso de lectura al principal autorizado.
Antes de entregar: ejecuta todas las celdas nuevas, comprueba que no haya errores y copia al documento Word los resultados reales de las consultas.


# 9. Conclusiones

En esta actividad se construyo una arquitectura Medallion funcional en Databricks.

Los datos originales se han conservado en Bronze, las reglas de calidad se han aplicado en Silver y las preguntas de negocio se han resuelto mediante tablas agregadas en Gold.

También se comprobo que el gobierno de datos no consiste únicamente en conceder permisos. Incluye documentar el significado de la información, clasificar los datos sensibles, conservar los registros rechazados y mantener la trazabilidad de las operaciones.

La actividad demuestra que una plataforma de datos fiable requiere tanto transformaciones técnicas como decisiones explícitas sobre la calidad y el uso de la información.

## Pregunta final

**Si encontramos películas con fechas incorrectas, ¿quién debería decidir si se corrigen, se descartan o se modifica la regla de validación?**

La decisión debe involucrar a los responsables del dato y del negocio. El equipo técnico implementa las reglas, pero no debería decidir por su cuenta qué información representa correctamente la realidad.